In [1]:
import torch
import os
from tianshou.data import Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv, SubprocVectorEnv
from tianshou.policy import DQNPolicy
from tianshou.trainer import OffpolicyTrainer
from dengue_envs.envs.dengue_diagnostics import DengueDiagnosticsEnv
from dengue_wrapper import DengueWrapper, CaseByCaseWrapper
from fcn_network import DengueNet
import numpy as np
from typing import Tuple
from tianshou.utils import TensorboardLogger
from torch.utils.tensorboard import SummaryWriter


C:\Users\segun\Documents\GitHub\Dengue_Diagnostics\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [3]:
DEVICE = "cuda"

LR = 1e-4
GAMMA = 0.99
N_STEP = 3
TARGET_UPDATE_FREQ = 1000

BUFFER_SIZE = 10000
BATCH_SIZE = 128

EPOCH = 15
STEP_PER_EPOCH = 10000

STEP_PER_COLLECT = 1000
UPDATE_PER_STEP = 0.1

EPS_TRAIN_START = 1.0
EPS_TRAIN_FINAL = 0.05
EPS_TRAIN_DECAY = 50000
EPS_TEST = 0.01
NUM_ENVS = 4
NUM_TEST_ENVS = 4

SEED = 850

In [4]:
WORLD_SIZE = 400
MIN_BORDER_DISTANCE = 50
MAX_RADIUS = 100
MIN_RADIUS = 50

def generate_random_center(size: int, margin: int) -> Tuple[int, int]:
    """Gera um par de coordenadas aleatórias dentro dos limites do mapa."""
    x = np.random.randint(margin, size - margin)
    y = np.random.randint(margin, size - margin)
    return int(x), int(y)

In [5]:
def make_env():
    """Função factory para criar o ambiente com randomização espacial e wrappers."""

    dengue_center = generate_random_center(WORLD_SIZE, MIN_BORDER_DISTANCE)
    chik_center = generate_random_center(WORLD_SIZE, MIN_BORDER_DISTANCE)
    dengue_radius = np.random.randint(MIN_RADIUS, MAX_RADIUS)
    chik_radius = np.random.randint(MIN_RADIUS, MAX_RADIUS)

    env = DengueDiagnosticsEnv(
        epilength=60,
        size=WORLD_SIZE,
        clinical_specificity=(0.5, 0.95),
        dengue_center=dengue_center,
        chik_center=chik_center,
        dengue_radius=dengue_radius,
        chik_radius=chik_radius
    )

    env = DengueWrapper(env)
    env = CaseByCaseWrapper(env)

    return env

In [6]:
print(f"VERIFICAÇÃO: O BUFFER_SIZE é {BUFFER_SIZE}")

VERIFICAÇÃO: O BUFFER_SIZE é 10000


In [7]:
if True:

    np.random.seed(SEED)
    torch.manual_seed(SEED)

    train_envs = SubprocVectorEnv([make_env for _ in range(NUM_ENVS)])
    test_envs = SubprocVectorEnv([make_env for _ in range(NUM_TEST_ENVS)])

    train_envs.seed(SEED)
    test_envs.seed(SEED)

    env = make_env()
    map_shape = env.observation_space.spaces["map"].shape
    action_shape = env.action_space.n

    net = DengueNet(map_shape, action_shape, device=DEVICE).to(DEVICE)
    optim = torch.optim.Adam(net.parameters(), lr=LR)

    policy = DQNPolicy(
        model=net,
        optim=optim,
        discount_factor=GAMMA,
        estimation_step=N_STEP,
        target_update_freq=TARGET_UPDATE_FREQ,
        action_space=env.action_space
    )

    print(f"DEBUG: TENTANDO USAR BUFFER_SIZE={BUFFER_SIZE}")

    buffer = VectorReplayBuffer(
        total_size=BUFFER_SIZE,
        buffer_num=NUM_ENVS,
        ignore_obs_next=True
    )

    train_collector = Collector(
        policy, train_envs, buffer, exploration_noise=True
    )
    test_collector = Collector(policy, test_envs)

    print("Forçando a coleta inicial de 100 passos para inicialização segura do buffer.")

    train_collector.collect(n_step=100, reset_before_collect=True)

    experiment_name = f"dqn_seed_{SEED}"

    log_path = os.path.join("logs", "experiment_15_epochs", experiment_name)
    writer = SummaryWriter(log_path)
    logger = TensorboardLogger(writer)

    def train_fn(epoch, env_step):
        if env_step <= EPS_TRAIN_DECAY:
            eps = EPS_TRAIN_START - env_step / EPS_TRAIN_DECAY * \
                  (EPS_TRAIN_START - EPS_TRAIN_FINAL)
        else:
            eps = EPS_TRAIN_FINAL
        policy.set_eps(eps)


    def test_fn(epoch, env_step):
        policy.set_eps(EPS_TEST)

    trainer = OffpolicyTrainer(
        policy=policy,
        train_collector=train_collector,
        test_collector=test_collector,
        max_epoch=EPOCH,
        step_per_epoch=STEP_PER_EPOCH,
        step_per_collect=STEP_PER_COLLECT,
        update_per_step=UPDATE_PER_STEP,
        episode_per_test=NUM_TEST_ENVS,
        batch_size=BATCH_SIZE,
        train_fn=train_fn,
        test_fn=test_fn,
        stop_fn=lambda mean_rewards: mean_rewards >= 100 ,
        logger=logger
    )

    print(f"Iniciando treinamento na {DEVICE}...")
    result = trainer.run()
    print("\n--- Resultado do Treinamento ---")
    print(result)

    torch.save(policy.state_dict(), f"dqn_dengue_policy_SEED_{SEED}.pth")
    print("Política salva em dqn_dengue_policy.pth")

DEBUG: TENTANDO USAR BUFFER_SIZE=10000
Forçando a coleta inicial de 100 passos para inicialização segura do buffer.
Iniciando treinamento na cuda...


Epoch #1: 10001it [14:11, 11.75it/s, env_step=10000, gradient_step=1000, len=260, n/ep=4, n/st=1000, rew=-287.00]                           


Epoch #1: test_reward: -258.350000 ± 148.096463, best_reward: -26.000000 ± 0.000000 in #0


Epoch #2: 10001it [17:14,  9.67it/s, env_step=20000, gradient_step=2000, len=260, n/ep=4, n/st=1000, rew=-351.85]                           


Epoch #2: test_reward: -150.050000 ± 189.100258, best_reward: -26.000000 ± 0.000000 in #0


...

Epoch #13: test_reward: -375.850000 ± 525.390067, best_reward: -26.000000 ± 0.000000 in #0


Epoch #14: 10001it [16:43,  9.97it/s, env_step=140000, gradient_step=14000, len=260, n/ep=4, n/st=1000, rew=-136.13]                           


Epoch #14: test_reward: -434.950000 ± 562.473210, best_reward: -26.000000 ± 0.000000 in #0


Epoch #15: 10001it [17:24,  9.57it/s, env_step=150000, gradient_step=15000, len=260, n/ep=4, n/st=1000, rew=-252.58]                           


Epoch #15: test_reward: -460.625000 ± 541.337383, best_reward: -26.000000 ± 0.000000 in #0

--- Resultado do Treinamento ---
InfoStats(gradient_step=15000, best_reward=-26.0, best_reward_std=0.0, train_step=150000, train_episode=576, test_step=37440, test_episode=144, timing=TimingStats(total_time=15855.91533446312, train_time=14943.432026147842, train_time_collect=3688.1474347114563, train_time_update=10810.144768238068, test_time=912.4833083152771, update_speed=10.037854740298732))
Política salva em dqn_dengue_policy.pth
